[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarmonizedMRI/raw2ismrmrd/blob/main/examples/scan_archive.ipynb)

In [ ]:
import importlib

if not importlib.util.find_spec('raw2ismrrmd'):
    %pip install raw2ismrmrd[examples]

# Scan Archive

In this notebook we are going through the following steps
- extract raw data from GE scan archive raw data file
- combine the raw data with the information from a pulseq seq file to create an ismrmrd file
- reconstruct the data

In [ ]:
from pathlib import Path

import h5py
import numpy as np
from einops import rearrange

from raw2ismrmrd.ismrmrd_from_sequence import ismrmrd_from_sequence
from raw2ismrmrd.utils import combine_ismrmrd_files

### Extract raw data from GE scan archive

To be able to get the raw data from a GE scan archive you need to have access to the `GERecon` toolbox
which requires a research agreement with GE. 

If you have got that then you can use the code provided in src/sa_to_npy.py or scr/sa_to_h5.py.

Use the `read_archive` function to extract the raw data and save it as a numpy array or h5 file.

Here we are going to download data which has already been extracted from the vendor specific raw data files ([10.5281/zenodo.20347379](https://zenodo.org/records/20347380)).


In [ ]:
import tempfile

import zenodo_get

tmp = tempfile.TemporaryDirectory()  # RAII, automatically cleaned up
data_folder = Path(tmp.name)
zenodo_get.download(
    record='20347379',
    retry_attempts=5,
    output_dir=data_folder,
)

### Combine raw data with pulseq information

In the next step we are going to combine the raw data from the scan archive file with the information about how the data
was acquired in the pulseq file. This only works if you used `labels` when creating the pulseq file to correctly identify 
at which k-space positions the data was acquired. Also other labels such as which echo number of which repetition number
the data is, is very helpful. Have a look here for an example of how this can be done: 
https://github.com/PTB-MR/mrseq/blob/main/src/mrseq/scripts/t1_t2_spiral_cmrf.py 

As mentioned above the scan archive data can either be saved in a numpy array or an h5 file. Here we use the h5 file, 
because it allows for readouts with different length (e.g. different number of readout points for imaging data and 
noise data). For the numpy array option you can simply exchange this code part:

```python
with h5py.File(data_folder + 't1_golden_radial.h5', 'r') as h5file:
    keys = sorted(h5file['frames'].keys())
    kdata_raw_list = [rearrange(h5file['frames'][key][:], 'readout coil -> coil readout') for key in keys]
```

with this:

```python
kdata_raw = np.load(data_folder + 't1_golden_radial.npy')
kdata_raw = rearrange(kdata_raw, 'readout acquisitions coils -> acquisitions coils readout')
kdata_raw_list = list(kdata_raw)
```

The raw data is passed to `ismrmrd_from_sequence` as a list of data objects of the shape `[coils readout]`. 
We use a list here because the different acquisitions do not have to have the same shape. 
E.g noise samples often have a different number of readout points compared to the data used for image reconstruction.

For certain sequences it is also important to know when each readout was acquired. These acquisition time stamps can 
also be added to the mrd file.

In [ ]:
with h5py.File(data_folder / 't1_golden_radial.h5', 'r') as h5file:
    keys = sorted(h5file['frames'].keys())
    kdata_raw_list = [rearrange(h5file['frames'][key][:], 'readout coil -> coil readout') for key in keys]


timestamps = np.load(data_folder / 't1_golden_radial_timestamps.npy')
timestamps_list = list(timestamps)

ismrmrd_from_sequence(
    kdata_raw_list,
    data_folder / 't1_golden_radial.seq',
    data_folder / 't1_golden_radial.mrd',
    timestamps_list,
    replace_mrd=True,
)

This scan was obtained with a golden angle radial readout. To make sure we have to correct trajectory we can combine the 
created mrd file with the trajectory information saved during the creation of this sequence.

In [ ]:
combine_ismrmrd_files(Path(data_folder / 't1_golden_radial.mrd'), Path(data_folder / 't1_golden_radial_header.h5'))

### Image reconstruction

Now we can check if everything worked by reconstructed the image data. 
Here we use [MRpro](https://github.com/PTB-MR/MRpro) for the image reconstruction.

This data is a 2D continuous golden angle acquisition after a single inversion pulse. The signal model can be described by transient steady state model (Look-Locker model). Fitting this model to different dynamic time frames reconstruction from the data yields a T1 map. 

For more information on this fit please see:
- Look D, Locker R (1970) Time Saving in Measurement of NMR and EPR Relaxation Times. Rev. Sci. Instrum 41 https://doi.org/10.1063/1.1684482
- Deichmann R, Haase A (1992) Quantification of T1 values by SNAPSHOT-FLASH NMR imaging. J. Magn. Reason. 612 http://doi.org/10.1016/0022-2364(92)90347-A
- Becker KM, Schulz-Menger J, Schaeffter T, Kolbitsch C (2019)  Simultaneous high-resolution cardiac T1 mapping and cine imaging using model-based iterative image reconstruction. Magn. Reason. Med. 81 https://doi.org/10.1002/mrm.27474



In [ ]:
import matplotlib.pyplot as plt
import torch
from cmap import Colormap
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import CsmData
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryIsmrmrd
from mrpro.operators import DictionaryMatchOp
from mrpro.operators.models import TransientSteadyStateWithPreparation

kdata = KData.from_file(data_folder + 't1_golden_radial_with_traj.mrd', KTrajectoryIsmrmrd())
csm = CsmData.from_kdata_inati(kdata[0, ..., 200:, :], downsampled_size=64, smoothing_width=9)

n_acq_per_image = 32
n_overlap = 16

split_indices = torch.arrange(kdata.shape[-2]).unfold(dimension=0, size=n_acq_per_image, step=n_overlap)
kdata_split = kdata[..., split_indices, :]

recon = DirectReconstruction(kdata_split, csm=csm)
idata = recon(kdata_split)

idat = idata.data.abs().numpy().squeeze()
fig, ax = plt.subplots(6, idat.shape[0] // 6, figsize=(2 * idata.shape[0] // 6, 6 * 2))
ax = ax.flatten()
for i in range(min(idat.shape[0], len(ax))):
    ax[i].imshow(idat[i, :, :], cmap='gray')
    ax[i].set_xticks([])
    ax[i].set_yticks([])

fig.suptitle('Qualitative images obtained after an inversion pulse')

sampling_time = idata.header.acquisition_time_stamp - kdata.header.acq_info.acquisition_time_stamp[0, 0, 0, 0, 0]
dictionary = DictionaryMatchOp(
    TransientSteadyStateWithPreparation(
        sampling_time=sampling_time.to(torch.float32),
        repetition_time=idata.header.tr,
        m0_scaling_preparation=-1,
        delay_after_preparation=0.019,
    ),
    index_of_scaling_parameter=0,
)
dictionary.append(
    torch.tensor(1.0),
    torch.linspace(0.1, 2.8, 500)[None, :, None],
    torch.deg2rad(torch.linspace(6, 10, 50))[None, None, :],
)
m0_match, t1_match, fa_match = dictionary(idata.data)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for cax in ax.flatten():
    cax.set_xticks([])
    cax.set_yticks([])

im = ax[0].imshow(m0_match.squeeze().abs().numpy(), cmap='grey')
fig.colorbar(im, ax=ax[0], label='M0')

im = ax[1].imshow(t1_match.squeeze().numpy(), vmin=0, vmax=2.0, cmap=Colormap('lipari').to_mpl())
fig.colorbar(im, ax=ax[1], label='T1 (s)')

im = ax[2].imshow(torch.rad2deg(fa_match).squeeze().numpy(), vmin=0, vmax=10, cmap='magma')
fig.colorbar(im, ax=ax[2], label='Flip Angle (°)')